# 03b · Supervised models: first run on v0, tuning log, imbalance experiment

Owner: Ayo (Y8–Y10). Random Forest and XGBoost from `src/models_supervised.py` on `data/processed/features_v0.parquet`. They train on **all** training rows (TEAM_PLAN 5.4) and are judged on the **val** split only; test is never looked at while tuning. Every run adds one row to `results/tables/tuning_log_supervised.csv`: parameters, val ROC-AUC, val PR-AUC, val recall at the top 1% of val user-days, and training time. The best run has the highest **val PR-AUC**.

On Mon Nov 9 (Y14) the same cells run on `features_v1.parquet`: change `DATASET` below.

Run with Restart & Run All. The log is rebuilt from scratch on every full run.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
sys.path.insert(0, str(ROOT))

import pandas as pd

from src import models_supervised as ms
from src.config import PROCESSED_DIR, TABLES_DIR

DATASET = 'v0'
df = pd.read_parquet(PROCESSED_DIR / f'features_{DATASET}.parquet')
log_path = TABLES_DIR / ms.TUNING_LOG
log_path.unlink(missing_ok=True)       # rebuilt from scratch on every Restart & Run All
X, y, cols = ms.get_model_matrix(df, split='train')
print(f'{DATASET}: {len(df):,} user-days, {len(cols)} features, {len(X):,} training rows '
      f'({int(y.sum())} malicious)')
df.groupby('split').label.agg(['size', 'sum']).rename(columns={'size': 'rows', 'sum': 'malicious'})

v0: 31,175 user-days, 7 features, 20,459 training rows (139 malicious)


,rows,malicious
split,,
test,5688,40
train,20459,139
val,5028,57


## Y8: first run with the card settings

Saved as `results/scores/<model>_v0.parquet` (the final `<model>.parquet` files come from v1 in Y15).

In [2]:
first = []
for name in ms.MODELS:
    log, _ = ms.tune_supervised(df, name, grid={}, dataset=DATASET)    # an empty grid = the card settings
    scores, info = ms.train_and_score_supervised(df, name, save_as=f'{name}_{DATASET}')
    first.append(log.assign(saved=str(info['path'].relative_to(ROOT))))
pd.concat(first)[['model', 'val_roc_auc', 'val_pr_auc', 'val_recall_top1pct', 'best_iteration', 'fit_seconds', 'saved']].round(4)

,model,val_roc_auc,val_pr_auc,val_recall_top1pct,best_iteration,fit_seconds,saved
0,random_forest,0.8084,0.0784,0.1930,None,0.43,results\scores\random_forest_v0.parquet
0,xgboost,0.9412,0.2961,0.2632,81,0.24,results\scores\xgboost_v0.parquet


## Y9: grid search

- **Random Forest:** `n_estimators` {200, 500} × `max_depth` {None, 10, 20} × `min_samples_leaf` {1, 5} × `max_features` {sqrt, 0.5} = 24 runs.
- **XGBoost:** `max_depth` {3, 6} × `learning_rate` {0.05, 0.1} × `subsample` {0.8, 1.0} × `min_child_weight` {1, 5} = 16 runs; the number of trees comes from early stopping on val.

In [3]:
rf_log, rf_best = ms.tune_supervised(df, 'random_forest', dataset=DATASET)
print('best Random Forest (val PR-AUC):', rf_best)
rf_log.sort_values('val_pr_auc', ascending=False).head(5).round(4)

best Random Forest (val PR-AUC): {'max_depth': 10, 'max_features': 0.5, 'min_samples_leaf': 5, 'n_estimators': 200}


,model,dataset,params,val_roc_auc,val_pr_auc,val_recall_top1pct,best_iteration,fit_seconds,n_features
14,random_forest,v0,"{""max_depth"": 10, ""max_features"": 0.5, ""min_sa...",0.9347,0.2733,0.2456,None,0.30,7
10,random_forest,v0,"{""max_depth"": 10, ""max_features"": ""sqrt"", ""min...",0.9425,0.2685,0.2807,None,0.30,7
11,random_forest,v0,"{""max_depth"": 10, ""max_features"": ""sqrt"", ""min...",0.9474,0.2658,0.2456,None,0.76,7
15,random_forest,v0,"{""max_depth"": 10, ""max_features"": 0.5, ""min_sa...",0.9362,0.2643,0.2456,None,0.72,7
13,random_forest,v0,"{""max_depth"": 10, ""max_features"": 0.5, ""min_sa...",0.9281,0.2444,0.2281,None,0.71,7


In [4]:
xgb_log, xgb_best = ms.tune_supervised(df, 'xgboost', dataset=DATASET)
print('best XGBoost (val PR-AUC):', xgb_best)
xgb_log.sort_values('val_pr_auc', ascending=False).head(5).round(4)

best XGBoost (val PR-AUC): {'learning_rate': 0.1, 'max_depth': 3, 'min_child_weight': 1, 'subsample': 0.8}


,model,dataset,params,val_roc_auc,val_pr_auc,val_recall_top1pct,best_iteration,fit_seconds,n_features
8,xgboost,v0,"{""learning_rate"": 0.1, ""max_depth"": 3, ""min_ch...",0.9519,0.3240,0.3333,162,0.19,7
10,xgboost,v0,"{""learning_rate"": 0.1, ""max_depth"": 3, ""min_ch...",0.9547,0.3218,0.2982,164,0.20,7
7,xgboost,v0,"{""learning_rate"": 0.05, ""max_depth"": 6, ""min_c...",0.9346,0.3049,0.2632,221,0.27,7
11,xgboost,v0,"{""learning_rate"": 0.1, ""max_depth"": 3, ""min_ch...",0.9493,0.3034,0.2807,165,0.17,7
4,xgboost,v0,"{""learning_rate"": 0.05, ""max_depth"": 6, ""min_c...",0.9412,0.2961,0.2632,81,0.15,7


## Y10: imbalance experiment

Three set-ups per model, all with the card settings: (a) no weighting, (b) class weights (`class_weight='balanced'` / `scale_pos_weight`), (c) SMOTE to a 10% minority **on the training rows only** (inside an imblearn Pipeline, so val and test are never resampled). Saved as `results/tables/imbalance.csv`.

In [5]:
imbalance = ms.imbalance_experiment(df, dataset=DATASET)
imbalance.round(4)

,model,setup,dataset,val_roc_auc,val_pr_auc,val_recall_top1pct,fit_seconds
0,random_forest,a_no_weighting,v0,0.8171,0.2424,0.2632,0.39
1,random_forest,b_class_weights,v0,0.8084,0.0784,0.1930,0.44
2,random_forest,c_smote_train_only,v0,0.8078,0.2451,0.3158,0.41
3,xgboost,a_no_weighting,v0,0.9643,0.4087,0.2982,0.24
4,xgboost,b_class_weights,v0,0.9412,0.2961,0.2632,0.17
5,xgboost,c_smote_train_only,v0,0.9599,0.4130,0.3860,0.38


## The tuning log and the T4 draft (Y13 step 1)

In [6]:
log = pd.read_csv(log_path)
print(f'{len(log)} runs in {log_path.relative_to(ROOT)}')
ms.hyperparameter_table().round(4)

42 runs in results\tables\tuning_log_supervised.csv


,model,defaults,grid,best_params,dataset,val_pr_auc,val_roc_auc,runs
0,random_forest,"{""n_estimators"": 300, ""class_weight"": ""balanced""}","{""n_estimators"": [200, 500], ""max_depth"": [nul...","{""max_depth"": 10, ""max_features"": 0.5, ""min_sa...",v0,0.2733,0.9347,25
1,xgboost,"{""n_estimators"": 1000, ""max_depth"": 6, ""learni...","{""max_depth"": [3, 6], ""learning_rate"": [0.05, ...","{""learning_rate"": 0.1, ""max_depth"": 3, ""min_ch...",v0,0.3240,0.9519,17


**Notes (Ayo):** what the v0 numbers say, which imbalance set-up to keep for the final models, and which settings to carry to v1. *Human-only: write your own reading here.*